# MNIST DPU Benchmark
Measures FPS, accuracy and power on the KV260 **FPGA DPU (B512)**.

**Task**: Classify handwritten digits (0-9) from 28x28 grayscale images.

**Run from Jupyter** (`http://<board-ip>:9090/lab`)

Prerequisites:
- Run `bash mnist/setup.sh` first to download MNIST dataset
- pynq venv active (Jupyter kernel handles this)

Expected: **~2863 FPS, 99% accuracy, 5.61W, 510 FPS/Watt**

In [ ]:
import os, time, gzip, numpy as np

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
DATA_DIR     = "/home/ubuntu/mnist_data"
N_WARMUP     = 5
N_BENCHMARK  = 100

# Smart path: check local models/ first, then pynq default
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("dpu_bench.ipynb"))
XMODEL_LOCAL = os.path.join(NOTEBOOK_DIR, "models", "dpu_mnist_classifier.xmodel")
XMODEL_PYNQ  = "/root/jupyter_notebooks/pynq-dpu/dpu_mnist_classifier.xmodel"
XMODEL = XMODEL_LOCAL if os.path.exists(XMODEL_LOCAL) else XMODEL_PYNQ

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"Using xmodel: {XMODEL}")
print(f"MNIST data dir: {DATA_DIR}")
print(f"Data exists: {os.path.exists(DATA_DIR)}")
print(f"Current power: {read_power_mw()/1000:.2f} W (idle)")

## Step 1 — Load MNIST Test Data
10,000 handwritten digit images. We benchmark on the first 100.

In [ ]:
def load_images(path):
    with gzip.open(path, 'rb') as f:
        f.read(16)  # skip header
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(-1, 28, 28)

def load_labels(path):
    with gzip.open(path, 'rb') as f:
        f.read(8)
        return np.frombuffer(f.read(), dtype=np.uint8)

images = load_images(f"{DATA_DIR}/t10k-images-idx3-ubyte.gz")
labels = load_labels(f"{DATA_DIR}/t10k-labels-idx1-ubyte.gz")

# DPU expects (1, 28, 28, 1) float32 — channel last
data = (images.astype(np.float32) / 255.0)[:, :, :, np.newaxis]

print(f"Loaded: {images.shape[0]} images, 28x28 pixels")
print(f"Data shape after preprocessing: {data.shape}  (N, H, W, C)")
print(f"Label sample: {labels[:10]}")

## Step 2 — Load DPU Overlay
`DpuOverlay('dpu.bit')` programs the FPGA with the B512 DPU (~10-20 seconds).

In [ ]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("dpu.bit")
print("DPU overlay loaded!")

## Step 3 — Load MNIST Model

In [ ]:
overlay.load_model(XMODEL)
dpu = overlay.runner
in_t  = dpu.get_input_tensors()
out_t = dpu.get_output_tensors()
print(f"Input shape:  {in_t[0].dims}  (28x28 grayscale image, channel last)")
print(f"Output shape: {out_t[0].dims}  (scores for digits 0-9)")

in_d  = [np.zeros(t.dims, dtype=np.float32) for t in in_t]
out_d = [np.zeros(t.dims, dtype=np.float32) for t in out_t]

## Step 4 — Warmup

In [ ]:
for i in range(N_WARMUP):
    in_d[0][0] = data[i]
    job = dpu.execute_async(in_d, out_d)
    dpu.wait(job)
print("Warmup done!")

## Step 5 — Benchmark with Accuracy + Power

In [ ]:
correct = 0
power_samples = []
start = time.time()
for i in range(N_BENCHMARK):
    in_d[0][0] = data[i]
    job = dpu.execute_async(in_d, out_d)
    dpu.wait(job)
    pred = int(np.argmax(out_d[0][0]))
    if pred == int(labels[i]):
        correct += 1
    if i % 10 == 0:
        power_samples.append(read_power_mw())
elapsed = time.time() - start
print("Done!")

## Step 6 — Results

In [ ]:
avg_power_w  = sum(power_samples) / len(power_samples) / 1000
fps          = N_BENCHMARK / elapsed
latency_ms   = elapsed / N_BENCHMARK * 1000
fps_per_watt = fps / avg_power_w
accuracy     = correct / N_BENCHMARK * 100

print("=" * 40)
print("MNIST DPU BENCHMARK RESULTS")
print("=" * 40)
print(f"Platform:    KV260 DPU (B512)")
print(f"Accuracy:    {accuracy:.1f}%  ({correct}/{N_BENCHMARK} correct)")
print(f"FPS:         {fps:.1f}")
print(f"Latency:     {latency_ms:.2f} ms/frame")
print(f"Power:       {avg_power_w:.2f} W")
print(f"FPS/Watt:    {fps_per_watt:.2f}")
print("=" * 40)
print(f"Expected:    ~2863 FPS, 99% accuracy, 5.61W, ~510 FPS/Watt")

del overlay